# FX Implied & Historical Correlation Dashboard

**Correlation Formulas:**

**Case 1: One common currency** (e.g., EURJPY vs EURGBP, both have EUR)
$$\rho = \frac{\sigma_{EURGBP}^2 + \sigma_{EURJPY}^2 - \sigma_{GBPJPY}^2}{2 \times \sigma_{EURJPY} \times \sigma_{EURGBP}}$$

**Case 2: No common currency** (e.g., EURJPY vs NZDUSD)
$$\rho = \frac{\sigma_{EURUSD}^2 + \sigma_{NZDJPY}^2 - \sigma_{USDJPY}^2 - \sigma_{EURNZD}^2}{2 \times \sigma_{EURJPY} \times \sigma_{NZDUSD}}$$

In [12]:
import numpy as np
import pandas as pd
from datetime import datetime
from itertools import combinations, permutations
import pytz

import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output

import bql

import warnings
warnings.filterwarnings('ignore')

# Configuration
CURRENCY_PAIRS = [
    'EURUSD', 'USDJPY', 'EURJPY', 'GBPUSD', 'EURGBP', 
    'USDCHF', 'EURCHF', 'AUDUSD', 'EURAUD', 'NZDUSD', 
    'USDCNH', 'USDZAR', 'EURZAR', 'JPYZAR'
]

# Extract all unique currencies
ALL_CURRENCIES = set()
for pair in CURRENCY_PAIRS:
    ALL_CURRENCIES.add(pair[:3])
    ALL_CURRENCIES.add(pair[3:])
ALL_CURRENCIES = sorted(list(ALL_CURRENCIES))

# Generate ALL possible currency pair combinations (both orderings)
ALL_POSSIBLE_PAIRS = set()
for c1 in ALL_CURRENCIES:
    for c2 in ALL_CURRENCIES:
        if c1 != c2:
            ALL_POSSIBLE_PAIRS.add(f"{c1}{c2}")
ALL_POSSIBLE_PAIRS = sorted(list(ALL_POSSIBLE_PAIRS))

TENORS = {
    '1W': '1 Week', '2W': '2 Weeks', '3W': '3 Weeks',
    '1M': '1 Month', '2M': '2 Months', '3M': '3 Months',
    '6M': '6 Months', '9M': '9 Months', '1Y': '1 Year', '2Y': '2 Years',
}

TERM_LOOKBACKS = {
    'Current': 0, 'T-3M': 63, 'T-6M': 126, 'T-12M': 252, 'T-2Y': 504, 'T-5Y': 1260,
}

BBG_SOURCE = 'BGNT'

CORRELATION_COLORSCALE = [
    [0.0, '#1a237e'], [0.25, '#42a5f5'], [0.5, '#ffffff'],
    [0.75, '#ef5350'], [1.0, '#b71c1c']
]

# Date setup
tz = pytz.timezone('Africa/Johannesburg')
today = pd.Timestamp('now', tz='UTC').tz_convert(tz)
yesterday_str = (today - pd.Timedelta(1, 'd')).strftime('%Y-%m-%d')

# Minimum required recent data (3 months = ~63 business days)
MIN_RECENT_DATA_DAYS = 63

#print(f"Currencies: {ALL_CURRENCIES}")
#print(f"Possible pair combinations: {len(ALL_POSSIBLE_PAIRS)}")

In [13]:
# Generate all tickers
all_tickers = []
for pair in ALL_POSSIBLE_PAIRS:
    for tenor in TENORS.keys():
        all_tickers.append(f"{pair}V{tenor} {BBG_SOURCE} Curncy")
        all_tickers.append(f"{pair}H{tenor} {BBG_SOURCE} Curncy")

all_tickers = sorted(list(set(all_tickers)))
#print(f"Total tickers to fetch: {len(all_tickers)}")

In [14]:
# Load ALL data from 2015-01-01 to yesterday
bq = bql.Service()

date_range = bq.func.range('2015-01-01', yesterday_str)
px_last = {'price': bq.data.px_last(dates=date_range, fill='prev')}

request = bql.Request(all_tickers, px_last)
response = bq.execute(request)

df_raw = response[0].df()
if 'CURRENCY' in df_raw.columns:
    df_raw.drop(columns=['CURRENCY'], inplace=True)

df_vol = df_raw.reset_index().pivot(index='DATE', columns='ID', values='price')
df_vol.index = pd.to_datetime(df_vol.index)
df_vol = df_vol[df_vol.index.weekday < 5].sort_index()

#print(f"Raw data: {df_vol.shape[0]} days, {df_vol.shape[1]} tickers")

In [15]:
# Filter columns: keep only those with at least 3 months of recent data
# Then forward fill remaining gaps

def validate_column(col: pd.Series, min_recent_days: int = 63) -> bool:
    """Check if column has at least min_recent_days of recent non-NaN data."""
    if col.isna().all():
        return False
    # Check last min_recent_days
    recent = col.tail(min_recent_days)
    # Need at least 50% non-NaN in recent period
    return recent.notna().sum() >= (min_recent_days * 0.5)

# Validate columns
valid_cols = [col for col in df_vol.columns if validate_column(df_vol[col], MIN_RECENT_DATA_DAYS)]
df_vol = df_vol[valid_cols]

# Forward fill remaining gaps
df_vol = df_vol.ffill()

#print(f"After validation: {df_vol.shape[0]} days, {df_vol.shape[1]} valid tickers")
#print(f"Date range: {df_vol.index.min().strftime('%Y-%m-%d')} to {df_vol.index.max().strftime('%Y-%m-%d')}")

In [16]:
# Volatility retrieval function

def get_vol(df: pd.DataFrame, pair: str, tenor: str, vol_type: str, idx: int = -1) -> float:
    """
    Get volatility value, trying both pair orderings.
    vol_type: 'V' for implied, 'H' for historical
    Returns np.nan if not found.
    """
    # Try standard ordering
    ticker = f"{pair}{vol_type}{tenor} {BBG_SOURCE} Curncy"
    if ticker in df.columns:
        val = df[ticker].iloc[idx]
        if pd.notna(val):
            return val
    
    # Try reversed ordering
    reversed_pair = pair[3:] + pair[:3]
    ticker_rev = f"{reversed_pair}{vol_type}{tenor} {BBG_SOURCE} Curncy"
    if ticker_rev in df.columns:
        val = df[ticker_rev].iloc[idx]
        if pd.notna(val):
            return val
    
    return np.nan

In [17]:
# Correlation calculation - handles both 1-common and 0-common currency cases

def calc_correlation(df: pd.DataFrame, pair1: str, pair2: str, 
                     tenor: str, vol_type: str, idx: int = -1) -> float:
    """
    Calculate correlation between two currency pairs.
    
    Case 1: One common currency (3-vol formula)
    For AB and AC (share A): ρ = (σ_AB² + σ_AC² - σ_BC²) / (2 × σ_AB × σ_AC)
    
    Case 2: No common currency (4-vol formula)
    For AB and CD: ρ = (σ_AD² + σ_CB² - σ_DB² - σ_AC²) / (2 × σ_AB × σ_CD)
    """
    if pair1 == pair2:
        return 1.0
    
    A, B = pair1[:3], pair1[3:]
    C, D = pair2[:3], pair2[3:]
    
    ccy1 = {A, B}
    ccy2 = {C, D}
    common = ccy1 & ccy2
    
    # Get the two main pair vols
    vol_AB = get_vol(df, pair1, tenor, vol_type, idx)
    vol_CD = get_vol(df, pair2, tenor, vol_type, idx)
    
    if pd.isna(vol_AB) or pd.isna(vol_CD) or vol_AB == 0 or vol_CD == 0:
        return np.nan
    
    if len(common) == 1:
        # Case 1: One common currency - use 3-vol formula
        # Find the unique currencies from each pair
        common_ccy = list(common)[0]
        unique_1 = list(ccy1 - common)[0]
        unique_2 = list(ccy2 - common)[0]
        
        # Cross pair is between the two unique currencies
        cross_pair = f"{unique_1}{unique_2}"
        vol_cross = get_vol(df, cross_pair, tenor, vol_type, idx)
        
        if pd.isna(vol_cross) or vol_cross == 0:
            return np.nan
        
        # ρ = (σ_AB² + σ_CD² - σ_cross²) / (2 × σ_AB × σ_CD)
        corr = (vol_AB**2 + vol_CD**2 - vol_cross**2) / (2 * vol_AB * vol_CD)
        
    elif len(common) == 0:
        # Case 2: No common currency - use 4-vol formula
        # For AB and CD: ρ = (σ_AD² + σ_CB² - σ_DB² - σ_AC²) / (2 × σ_AB × σ_CD)
        
        pair_AD = f"{A}{D}"
        pair_CB = f"{C}{B}"
        pair_DB = f"{D}{B}"
        pair_AC = f"{A}{C}"
        
        vol_AD = get_vol(df, pair_AD, tenor, vol_type, idx)
        vol_CB = get_vol(df, pair_CB, tenor, vol_type, idx)
        vol_DB = get_vol(df, pair_DB, tenor, vol_type, idx)
        vol_AC = get_vol(df, pair_AC, tenor, vol_type, idx)
        
        # Check if all 4 vols are available and non-zero
        if any(pd.isna(v) or v == 0 for v in [vol_AD, vol_CB, vol_DB, vol_AC]):
            return np.nan
        
        corr = (vol_AD**2 + vol_CB**2 - vol_DB**2 - vol_AC**2) / (2 * vol_AB * vol_CD)
        
    else:
        # len(common) == 2 means same pair
        return 1.0
    
    # If correlation is outside valid range, return NaN (indicates missing/invalid data)
    if corr < -1.01 or corr > 1.01:
        return np.nan
    # Small tolerance for numerical precision
    return np.clip(corr, -1, 1)


def build_corr_matrix(df: pd.DataFrame, pairs: list, tenor: str, 
                      vol_type: str = 'V', idx: int = -1) -> pd.DataFrame:
    """Build correlation matrix."""
    n = len(pairs)
    matrix = np.full((n, n), np.nan)
    
    for i in range(n):
        for j in range(n):
            matrix[i, j] = calc_correlation(df, pairs[i], pairs[j], tenor, vol_type, idx)
    
    return pd.DataFrame(matrix, index=pairs, columns=pairs)

In [18]:
# Test the correlation calculation
#print("Testing correlation calculations (1M implied):")
#print(f"USDZAR vs EURZAR (1 common: ZAR): {calc_correlation(df_vol, 'USDZAR', 'EURZAR', '1M', 'V'):.4f}")
#print(f"EURJPY vs EURGBP (1 common: EUR): {calc_correlation(df_vol, 'EURJPY', 'EURGBP', '1M', 'V'):.4f}")
#print(f"EURUSD vs NZDUSD (1 common: USD): {calc_correlation(df_vol, 'EURUSD', 'NZDUSD', '1M', 'V'):.4f}")
#print(f"EURJPY vs NZDUSD (0 common): {calc_correlation(df_vol, 'EURJPY', 'NZDUSD', '1M', 'V')}")
#print(f"EURZAR vs GBPUSD (0 common): {calc_correlation(df_vol, 'EURZAR', 'GBPUSD', '1M', 'V')}")
#print()
#print("Checking problematic pairs (should be NaN if cross vol missing):")
#print(f"NZDUSD vs USDZAR: {calc_correlation(df_vol, 'NZDUSD', 'USDZAR', '1M', 'V')} (needs NZDZAR)")
#print(f"NZDUSD vs EURZAR: {calc_correlation(df_vol, 'NZDUSD', 'EURZAR', '1M', 'V')} (needs NZDZAR, NZDEUR)")
#print(f"NZDUSD vs JPYZAR: {calc_correlation(df_vol, 'NZDUSD', 'JPYZAR', '1M', 'V')} (needs NZDZAR, NZDJPY)")
#print(f"USDCNH vs EURJPY: {calc_correlation(df_vol, 'USDCNH', 'EURJPY', '1M', 'V')} (needs EURCNH, JPYCNH)")
#print(f"USDCNH vs EURGBP: {calc_correlation(df_vol, 'USDCNH', 'EURGBP', '1M', 'V')} (needs GBPCNH, EURCNH)")
#print()
#print("Checking if required cross pair vols exist:")
#print(f"NZDZAR vol: {get_vol(df_vol, 'NZDZAR', '1M', 'V')}")
#print(f"EURCNH vol: {get_vol(df_vol, 'EURCNH', '1M', 'V')}")
#print(f"JPYCNH vol: {get_vol(df_vol, 'JPYCNH', '1M', 'V')}")
#print(f"GBPCNH vol: {get_vol(df_vol, 'GBPCNH', '1M', 'V')}")

In [19]:
# Visualization Functions

def get_percentile_thresholds(matrix: pd.DataFrame, low_pct: float = 20, high_pct: float = 80) -> tuple:
    """
    Calculate percentile thresholds for highlighting.
    Excludes diagonal (1.0) and NaN values.
    """
    # Get off-diagonal values only
    vals = []
    for i, row in enumerate(matrix.index):
        for j, col in enumerate(matrix.columns):
            if i != j:  # Exclude diagonal
                val = matrix.loc[row, col]
                if pd.notna(val):
                    vals.append(val)
    
    if len(vals) == 0:
        return -1, 1
    
    low_thresh = np.percentile(vals, low_pct)
    high_thresh = np.percentile(vals, high_pct)
    return low_thresh, high_thresh


def create_heatmap(corr_matrix: pd.DataFrame, title: str) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index,
        colorscale=CORRELATION_COLORSCALE, zmin=-1, zmax=1,
        colorbar=dict(title='ρ', thickness=20, len=0.8),
        hovertemplate='%{y} vs %{x}<br>ρ = %{z:.3f}<extra></extra>'
    ))
    
    # Calculate percentile thresholds
    low_thresh, high_thresh = get_percentile_thresholds(corr_matrix, 20, 80)
    
    annotations = []
    for i, row in enumerate(corr_matrix.index):
        for j, col in enumerate(corr_matrix.columns):
            val = corr_matrix.loc[row, col]
            if pd.notna(val):
                # Determine text color and style
                if i != j and val >= high_thresh:  # Top 20% (80th percentile+)
                    color = '#FFD700'  # Bold Yellow/Gold
                    bold = True
                elif i != j and val <= low_thresh:  # Bottom 20%
                    color = '#00AA00'  # Bold Green
                    bold = True
                else:
                    color = 'white' if abs(val) > 0.5 else 'black'
                    bold = False
                
                text = f'<b>{val:.2f}</b>' if bold else f'{val:.2f}'
                annotations.append(dict(x=col, y=row, text=text,
                                        showarrow=False, font=dict(color=color, size=9)))
    fig.update_layout(annotations=annotations)
    
    fig.update_layout(
        title=dict(text=f"{title}<br><sup><span style='color:#FFD700;'>Yellow Text: Top Quintile</span> | <span style='color:#00AA00;'>Green Text: Bottom Quintile</span></sup>", 
                   font=dict(size=18), x=0.5),
        xaxis=dict(tickfont=dict(size=11), tickangle=45),
        yaxis=dict(tickfont=dict(size=11), autorange='reversed'),
        width=900, height=800, plot_bgcolor='white',
        margin=dict(l=100, r=50, t=100, b=100)
    )
    return fig


def create_diff_heatmap(impl: pd.DataFrame, hist: pd.DataFrame, title: str) -> go.Figure:
    diff = impl - hist
    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=diff.values, x=diff.columns, y=diff.index,
        colorscale='RdBu_r', zmid=0, zmin=-0.5, zmax=0.5,
        colorbar=dict(title='Δρ', thickness=20, len=0.8),
        hovertemplate='%{y} vs %{x}<br>Δρ = %{z:+.3f}<extra></extra>'
    ))
    
    # Calculate percentile thresholds for difference
    low_thresh, high_thresh = get_percentile_thresholds(diff, 20, 80)
    
    annotations = []
    for i, row in enumerate(diff.index):
        for j, col in enumerate(diff.columns):
            val = diff.loc[row, col]
            if pd.notna(val):
                # Determine text color and style
                if i != j and val >= high_thresh:  # Top 20% (implied >> historical)
                    color = '#FFD700'  # Bold Yellow/Gold
                    bold = True
                elif i != j and val <= low_thresh:  # Bottom 20% (historical >> implied)
                    color = '#00AA00'  # Bold Green
                    bold = True
                else:
                    color = 'white' if abs(val) > 0.25 else 'black'
                    bold = False
                
                text = f'<b>{val:+.2f}</b>' if bold else f'{val:+.2f}'
                annotations.append(dict(x=col, y=row, text=text,
                                        showarrow=False, font=dict(color=color, size=9)))
    fig.update_layout(annotations=annotations)
    
    fig.update_layout(
        title=dict(text=f"{title}<br><sup><span style='color:#FFD700;'>Yellow Text: Top Quintile</span> | <span style='color:#00AA00;'>Green Text: Bottom Quintile</span></sup>", 
                   font=dict(size=18), x=0.5),
        xaxis=dict(tickfont=dict(size=11), tickangle=45),
        yaxis=dict(tickfont=dict(size=11), autorange='reversed'),
        width=900, height=800, plot_bgcolor='white',
        margin=dict(l=100, r=50, t=100, b=100)
    )
    return fig

In [20]:
# Time Series Functions

def calc_corr_timeseries(df: pd.DataFrame, pair1: str, pair2: str, 
                          tenor: str, start: str, end: str) -> pd.DataFrame:
    """Calculate correlation time series."""
    mask = (df.index >= start) & (df.index <= end)
    df_filt = df.loc[mask]
    
    results = []
    for i in range(len(df_filt)):
        impl = calc_correlation(df_filt, pair1, pair2, tenor, 'V', i)
        hist = calc_correlation(df_filt, pair1, pair2, tenor, 'H', i)
        
        results.append({
            'date': df_filt.index[i],
            'implied': impl,
            'historical': hist,
            'spread': impl - hist if pd.notna(impl) and pd.notna(hist) else np.nan
        })
    
    return pd.DataFrame(results).set_index('date')


def create_corr_chart(ts: pd.DataFrame, pair1: str, pair2: str, tenor: str) -> go.Figure:
    # Dynamic y-axis range
    all_vals = pd.concat([ts['implied'], ts['historical']]).dropna()
    if len(all_vals) > 0:
        y_min = max(all_vals.min() - 0.05, -1)
        y_max = min(all_vals.max() + 0.05, 1)
    else:
        y_min, y_max = -1, 1
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts.index, y=ts['implied'], mode='lines',
                              name='Implied', line=dict(color='#1976d2', width=2)))
    fig.add_trace(go.Scatter(x=ts.index, y=ts['historical'], mode='lines',
                              name='Historical', line=dict(color='#d32f2f', width=2)))
    fig.update_layout(
        title=dict(text=f'{pair1} vs {pair2} Correlation ({tenor})', font=dict(size=16), x=0.5),
        yaxis=dict(title='Correlation', range=[y_min, y_max]),
        legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
        width=1100, height=400, plot_bgcolor='white', hovermode='x unified',
        margin=dict(l=60, r=40, t=60, b=80)
    )
    fig.update_xaxes(showgrid=True, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridcolor='lightgray')
    return fig


def create_spread_chart(ts: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)
    fig.add_trace(go.Scatter(x=ts.index, y=ts['spread'], mode='lines',
                              line=dict(color='#388e3c', width=2),
                              fill='tozeroy', fillcolor='rgba(56, 142, 60, 0.2)'))
    
    spread_vals = ts['spread'].dropna()
    if len(spread_vals) > 0:
        spread_max = max(abs(spread_vals.min()), abs(spread_vals.max())) * 1.2
        spread_max = max(spread_max, 0.05)
    else:
        spread_max = 0.5
    
    fig.update_layout(
        title=dict(text='Spread (Implied - Historical)', font=dict(size=14), x=0.5),
        xaxis=dict(title='Date'), yaxis=dict(title='Spread', range=[-spread_max, spread_max]),
        width=1100, height=250, plot_bgcolor='white', hovermode='x unified',
        margin=dict(l=60, r=40, t=50, b=50), showlegend=False
    )
    fig.update_xaxes(showgrid=True, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridcolor='lightgray')
    return fig


def create_term_structure(df: pd.DataFrame, pair1: str, pair2: str, lookback: str = 'Current') -> go.Figure:
    """Create term structure with lookback option."""
    lookback_days = TERM_LOOKBACKS.get(lookback, 0)
    idx = -1 - lookback_days
    if abs(idx) > len(df):
        idx = 0
    
    date_used = df.index[idx].strftime('%Y-%m-%d')
    
    labels, impl_corrs, hist_corrs = [], [], []
    for tenor, label in TENORS.items():
        impl_corrs.append(calc_correlation(df, pair1, pair2, tenor, 'V', idx))
        hist_corrs.append(calc_correlation(df, pair1, pair2, tenor, 'H', idx))
        labels.append(label)
    
    # Dynamic y-axis
    all_vals = [v for v in impl_corrs + hist_corrs if pd.notna(v)]
    if len(all_vals) > 0:
        y_min = max(min(all_vals) - 0.05, -1)
        y_max = min(max(all_vals) + 0.05, 1)
    else:
        y_min, y_max = -1, 1
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=labels, y=impl_corrs, mode='lines+markers',
                              name='Implied', line=dict(color='#1976d2', width=2), marker=dict(size=8)))
    fig.add_trace(go.Scatter(x=labels, y=hist_corrs, mode='lines+markers',
                              name='Historical', line=dict(color='#d32f2f', width=2), marker=dict(size=8)))
    fig.update_layout(
        title=dict(text=f'{pair1} vs {pair2}: Term Structure ({lookback} - {date_used})', font=dict(size=16), x=0.5),
        xaxis=dict(title='Tenor'), yaxis=dict(title='Correlation', range=[y_min, y_max]),
        legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
        width=1100, height=400, plot_bgcolor='white',
        margin=dict(l=60, r=40, t=60, b=80)
    )
    fig.update_xaxes(showgrid=True, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridcolor='lightgray')
    return fig

In [21]:
# Dashboard

class Dashboard:
    def __init__(self, df, pairs, tenors):
        self.df = df
        self.pairs = pairs
        self.tenors = tenors
        
        min_dt = df.index.min()
        max_dt = df.index.max()
        
        self.start_date = widgets.DatePicker(description='Start:', value=pd.to_datetime('2024-01-01'))
        self.end_date = widgets.DatePicker(description='End:', value=max_dt)
        self.tenor_dd = widgets.Dropdown(options=[(v, k) for k, v in tenors.items()], value='1M', description='Tenor:')
        self.type_radio = widgets.RadioButtons(options=['Implied', 'Historical', 'Difference'], value='Implied', description='Type:')
        self.pair1_dd = widgets.Dropdown(options=pairs, value='USDZAR', description='Pair 1:')
        self.pair2_dd = widgets.Dropdown(options=pairs, value='EURZAR', description='Pair 2:')
        self.term_lookback_dd = widgets.Dropdown(
            options=list(TERM_LOOKBACKS.keys()), value='Current', description='Term Date:')
        
        self.matrix_out = widgets.Output()
        self.ts_out = widgets.Output()
        
        self.tenor_dd.observe(self.update_all, names='value')
        self.type_radio.observe(self.update_matrix, names='value')
        self.pair1_dd.observe(self.update_ts, names='value')
        self.pair2_dd.observe(self.update_ts, names='value')
        self.start_date.observe(self.update_ts, names='value')
        self.end_date.observe(self.update_ts, names='value')
        self.term_lookback_dd.observe(self.update_ts, names='value')
        
        self.layout = widgets.VBox([
            widgets.HTML(f"<h2 style='color:#1a237e;'>FX Correlation Dashboard</h2><i>Data: {min_dt.strftime('%Y-%m-%d')} to {max_dt.strftime('%Y-%m-%d')}</i>"),
            widgets.HTML("<hr><b>Correlation Matrix:</b>"),
            widgets.HBox([self.tenor_dd, self.type_radio]),
            self.matrix_out,
            widgets.HTML("<br><hr><b>Time Series & Term Structure:</b>"),
            widgets.HBox([self.pair1_dd, self.pair2_dd]),
            widgets.HBox([self.start_date, self.end_date, self.term_lookback_dd]),
            self.ts_out
        ])
    
    def update_matrix(self, change=None):
        with self.matrix_out:
            clear_output(wait=True)
            tenor = self.tenor_dd.value
            typ = self.type_radio.value.lower()
            
            if typ == 'difference':
                impl = build_corr_matrix(self.df, self.pairs, tenor, 'V')
                hist = build_corr_matrix(self.df, self.pairs, tenor, 'H')
                create_diff_heatmap(impl, hist, f'Implied - Historical ({TENORS[tenor]})').show()
            else:
                vol_type = 'V' if typ == 'implied' else 'H'
                corr = build_corr_matrix(self.df, self.pairs, tenor, vol_type)
                create_heatmap(corr, f'{typ.title()} Correlation ({TENORS[tenor]})').show()
    
    def update_ts(self, change=None):
        if self.pair1_dd.value == self.pair2_dd.value:
            return
        with self.ts_out:
            clear_output(wait=True)
            start = self.start_date.value.strftime('%Y-%m-%d') if self.start_date.value else '2024-01-01'
            end = self.end_date.value.strftime('%Y-%m-%d') if self.end_date.value else yesterday_str
            
            ts = calc_corr_timeseries(self.df, self.pair1_dd.value, self.pair2_dd.value, 
                                       self.tenor_dd.value, start, end)
            if len(ts) > 0:
                create_corr_chart(ts, self.pair1_dd.value, self.pair2_dd.value, self.tenor_dd.value).show()
                create_spread_chart(ts).show()
            
            create_term_structure(self.df, self.pair1_dd.value, self.pair2_dd.value, 
                                   self.term_lookback_dd.value).show()
    
    def update_all(self, change=None):
        self.update_matrix()
        self.update_ts()
    
    def show(self):
        display(self.layout)
        self.update_all()

In [22]:
# Run
dash = Dashboard(df_vol, CURRENCY_PAIRS, TENORS)
dash.show()